In [0]:
df = spark.read.format("csv").option("header",True).load("/Volumes/practice/default/data_sample/titanic_data.csv")

In [0]:
df.display()

#1 Calculate Running total

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
ex1 = (
    df.groupBy("embark_town").agg(
        sum(col("survived").cast('int')).alias("Survived")
    )
    .orderBy("Survived")
)
ex1.display()

In [0]:
## Window

window_spec = Window.orderBy("embark_town").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)

run_total = ex1.withColumn(
    "Running_tot",
    sum("Survived").over(window_spec)
)
run_total.display()

In [0]:
df.createOrReplaceTempView("sql_table")

In [0]:
%sql
-- Same in SQL

with cte1 as (
select 
embark_town,
sum(cast(survived as int)) as Survived
from sql_table
group by 1
order by 1)
select
*,
sum(survived) over(order by embark_town) as Running_tot
from cte1

#02 Schema Inferencing

In [0]:
df.display(10)

In [0]:
from pyspark.sql.types import *
schema = StructType([
    StructField("survived",IntegerType()),
    StructField("pclass",IntegerType()),
    StructField("sex",StringType()),
    StructField("age",IntegerType()),
    StructField("sibsp",IntegerType()),
    StructField("parch",IntegerType()),
    StructField("fare",FloatType()),
    StructField("embarked",StringType()),
    StructField("class",StringType()),
    StructField("who",StringType()),
    StructField("adult_male",BooleanType()),
    StructField("deck",StringType()),
    StructField("embark_town",StringType()),
    StructField("alive",StringType()),
    StructField("alone",BooleanType()),
])

df = spark.read.format("csv").option("header",True).schema(schema).load("/Volumes/practice/default/data_sample/titanic_data.csv")

In [0]:
df.display()

In [0]:
df.write.mode("overwrite").option("overwriteSchema",True).saveAsTable("practice.default.titanic_practice")

# 03 Implements SCD-Type 2

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
schema = StructType([
    StructField("address",StringType()),
    StructField("city",StringType()),
    StructField("contact",StringType()),
    StructField("cust_type",StringType()),
    StructField("customer_id",IntegerType()),
    StructField("service_cd",StringType()),
    StructField("state",StringType())
])

In [0]:
## data load

df = spark.read.option('multiline',True).schema(schema).json('/Volumes/practice/default/data_sample/scd_examples.json')
df = df.withColumn(
    "contact",
    trim(col("contact")).cast('bigint')
)
df.display()

In [0]:
## adding effective date and expiry date

df_eff_dt = df.withColumn(
    "eff_dt",
    lit("2026-05-18").cast("date")
)
df_eff_dt = df_eff_dt.withColumn(
    "exp_dt",
    lit("9999-12-31").cast("date")
)
df_eff_dt.display()

In [0]:
## saving as delta table

df_eff_dt.write.option("overwriteSchema",True).mode("overwrite").saveAsTable("practice.default.cust_type2_table")

In [0]:
## creating scenerio where we are getting new and updated data
new_file = spark.read.option('multiline',True).schema(schema).json('/Volumes/practice/default/data_sample/case3.json')
hash_cols = ['address','city','contact','cust_type','customer_id','service_cd','state']

new_file = new_file.withColumn(
    "contact",
    trim(col("contact")).cast('bigint')
) \
.withColumn(
    "hashkey",
    sha2(
        concat_ws(
            "||",
            *[
                upper(trim(col(c).cast('string')))
                for c in hash_cols
            ]
        ),
        256
    )
)
new_file.display()

In [0]:
## finding new records
from delta.tables import DeltaTable
path = 'practice.default.cust_type2_table'
src_delta = DeltaTable.forName(spark,path)
src_tab = src_delta.toDF()
src_tab = (
    src_tab.withColumn(
        "hashkey",
        sha2(
            concat_ws(
                "||",
                *[
                    upper(trim(col(c).cast('string')))
                    for c in hash_cols
                ]
            ),
            256
        )
    )
)
src_tab.display()

In [0]:
changed_df = src_tab.alias("a").filter((col("exp_dt") > current_date()) & (col("eff_dt") < col("exp_dt"))).join(
    new_file.alias("b"),
    on=[(col("a.customer_id") == col("b.customer_id"))],
    how="inner"
) \
.filter(col("a.hashkey") != col("b.hashkey")).select("b.*")
changed_df.write.mode('overwrite').saveAsTable('practice.default.changed_df')
changed_df.display()

In [0]:
# Identify changed records

changed_df = src_tab.alias("a").filter((col("exp_dt") > current_date()) & (col("eff_dt") < col("exp_dt"))).join(
    new_file.alias("b"),
    on=[(col("a.customer_id") == col("b.customer_id"))],
    how="inner"
) \
.filter((col("a.hashkey") != col("b.hashkey")) & (col("a.cust_type") != col("b.cust_type")) & (col("a.service_cd") != col("b.service_cd"))).select("b.*")
changed_df.write.mode('overwrite').saveAsTable('practice.default.changed_df')
changed_df.display()

In [0]:
%sql
RESTORE TABLE practice.default.cust_type2_table TO VERSION AS OF 4;

In [0]:
%sql
select * from practice.default.cust_type2_table
order by customer_id

In [0]:
## Insert new + updated records
new_df = new_file.alias("a").join(
    src_tab.alias("b"),
    on="customer_id",
    how="left_anti"
)
new_df.display()

In [0]:
updated_records_df = changed_df
updated_records_df.display()

In [0]:
## Expire existing active records

src_delta.alias("a").merge(
    changed_df.alias("b"),
    "a.customer_id = b.customer_id and a.eff_dt < a.exp_dt and a.exp_dt > current_date"
).whenMatchedUpdate(
    set = {
       "exp_dt":current_date()
    }
).execute()

In [0]:
updated_records_df = spark.table('practice.default.changed_df')

final_df = new_df.unionByName(updated_records_df) \
    .withColumn(
        "eff_dt",
        current_date()
    ) \
    .withColumn(
        "exp_dt",
        lit("9999-12-31").cast("date")
    ) \
    .drop("hashkey")
final_df.display()

In [0]:
final_df.write.mode('append').saveAsTable(path)

In [0]:
%sql
select * from practice.default.cust_type2_table

In [0]:
%sql
select * from practice.default.cust_type2_table
where eff_dt < exp_dt and exp_dt > current_date